# U-Net GLS Lesion Segmentation: Standalone Test Notebook

This notebook is a standalone evaluator/inference runner for the U-Net lesion segmentation model.

It includes:
- all imports
- repo root resolution
- robust checkpoint loading
- full test-set Dice/IoU evaluation
- random qualitative visualization
- optional single-image inference

In [ ]:
# 1) Imports
import os
import sys
import random
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
# 2) Resolve repo root and import model
repo_root = Path.cwd().resolve()
if repo_root.name == 'image_seg':
    repo_root = repo_root.parents[1]
elif repo_root.name == 'src':
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.unet import UNet

print(f'Repo root: {repo_root}')
print('Loaded UNet from src/models/unet.py')

In [ ]:
# 3) Configuration
SEED = 42
IMAGE_SIZE = 512
BATCH_SIZE = 8
NUM_WORKERS = 0  # Increase on Linux/cloud if needed
BASE_CHANNELS = 64
THRESHOLD = 0.5

TEST_IMAGE_DIR = repo_root / 'data' / 'unet_image_seg_real' / 'test' / 'image'
TEST_MASK_DIR = repo_root / 'data' / 'unet_image_seg_real' / 'test' / 'mask'

# Optional manual override; set to an explicit checkpoint file if needed.
CHECKPOINT_PATH = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Test image dir: {TEST_IMAGE_DIR}')
print(f'Test mask dir:  {TEST_MASK_DIR}')

In [ ]:
# 4) Utilities: file listing, dataset, metrics, checkpoint finder
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def list_image_files(folder: Path):
    if not folder.exists():
        raise FileNotFoundError(f'Folder not found: {folder}')
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS])

def denorm(img_t: torch.Tensor) -> torch.Tensor:
    mean = MEAN.to(img_t.device)
    std = STD.to(img_t.device)
    return (img_t * std + mean).clamp(0, 1)

def dice_score_from_logits(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    pred = (probs > threshold).float()
    num = 2.0 * (pred * targets).sum(dim=(1, 2, 3)) + eps
    den = pred.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps
    return (num / den).mean().item()

def iou_score_from_logits(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    pred = (probs > threshold).float()
    inter = (pred * targets).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - inter
    return ((inter + eps) / (union + eps)).mean().item()

class SegTestDataset(Dataset):
    def __init__(self, image_dir: Path, mask_dir: Path, image_size: int = 512):
        self.image_files = list_image_files(image_dir)
        self.mask_files = list_image_files(mask_dir)
        if len(self.image_files) != len(self.mask_files):
            raise RuntimeError('Image/mask file count mismatch in test folders.')
        self.image_size = image_size

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        mask_path = self.mask_files[idx]

        img = Image.open(img_path).convert('RGB').resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = Image.open(mask_path).convert('L').resize((self.image_size, self.image_size), Image.NEAREST)

        img_np = np.asarray(img, dtype=np.float32) / 255.0
        mask_np = (np.asarray(mask, dtype=np.float32) / 255.0 > 0.5).astype(np.float32)

        img_t = torch.from_numpy(img_np).permute(2, 0, 1).contiguous()
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).contiguous()
        img_t = (img_t - MEAN) / STD

        return img_t, mask_t, img_path.name

def find_checkpoint(repo_root: Path) -> Path:
    if CHECKPOINT_PATH is not None:
        p = Path(CHECKPOINT_PATH)
        if not p.exists():
            raise FileNotFoundError(f'Manual CHECKPOINT_PATH not found: {p}')
        return p

    preferred = [
        repo_root / 'src' / 'models' / 'checkpoints' / 'unet_gls_lesion_best.pth',
        repo_root / 'src' / 'models' / 'checkpoints' / 'unet_gls_lesion_epoch050.pth',
    ]
    for p in preferred:
        if p.exists():
            return p

    candidates = sorted((repo_root / 'src' / 'models').rglob('*.pth'))
    unet_like = [p for p in candidates if 'unet' in p.name.lower()]
    if unet_like:
        return unet_like[-1]

    msg = 'No checkpoint found. Set CHECKPOINT_PATH manually. Searched under src/models/**/*.pth'
    raise FileNotFoundError(msg)

In [ ]:
# 5) Load model checkpoint
checkpoint_path = find_checkpoint(repo_root)
print(f'Using checkpoint: {checkpoint_path}')

net = UNet(in_channels=3, out_channels=1, base_channels=BASE_CHANNELS).to(device)
ckpt = torch.load(checkpoint_path, map_location=device)

if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
    state_dict = ckpt['model_state_dict']
else:
    state_dict = ckpt

net.load_state_dict(state_dict, strict=True)
net.eval()

if isinstance(ckpt, dict):
    print(f"Checkpoint keys: {list(ckpt.keys())[:10]}")
print('Model loaded and ready for inference.')

In [ ]:
# 6) Build test dataloader
test_dataset = SegTestDataset(TEST_IMAGE_DIR, TEST_MASK_DIR, image_size=IMAGE_SIZE)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda')
)

print(f'Test samples: {len(test_dataset)}')

In [ ]:
# 7) Evaluate on full test set
@torch.no_grad()
def evaluate_model(model, loader, threshold=0.5):
    model.eval()
    total_dice = 0.0
    total_iou = 0.0
    n = 0

    for images, masks, _ in tqdm(loader, desc='Evaluating'):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        logits = model(images)

        bs = images.size(0)
        total_dice += dice_score_from_logits(logits, masks, threshold=threshold) * bs
        total_iou += iou_score_from_logits(logits, masks, threshold=threshold) * bs
        n += bs

    return total_dice / n, total_iou / n

test_dice, test_iou = evaluate_model(net, test_loader, threshold=THRESHOLD)
print(f'Test Dice: {test_dice:.4f}')
print(f'Test IoU:  {test_iou:.4f}')

In [ ]:
# 8) Visualize random predictions
@torch.no_grad()
def show_random_predictions(model, dataset, num_samples=6, threshold=0.5):
    model.eval()
    n = len(dataset)
    if n == 0:
        print('Dataset is empty.')
        return

    k = min(num_samples, n)
    indices = random.sample(range(n), k)

    fig, axs = plt.subplots(k, 3, figsize=(12, 4 * k))
    if k == 1:
        axs = np.expand_dims(axs, axis=0)

    for row, idx in enumerate(indices):
        img_t, mask_t, fname = dataset[idx]
        logits = model(img_t.unsqueeze(0).to(device))
        pred = (torch.sigmoid(logits).squeeze().cpu().numpy() > threshold).astype(np.float32)

        img_np = denorm(img_t).permute(1, 2, 0).cpu().numpy()
        gt_np = mask_t.squeeze(0).cpu().numpy()

        axs[row, 0].imshow(img_np)
        axs[row, 0].set_title(f'Input: {fname}')
        axs[row, 0].axis('off')

        axs[row, 1].imshow(gt_np, cmap='gray')
        axs[row, 1].set_title('Ground Truth')
        axs[row, 1].axis('off')

        axs[row, 2].imshow(pred, cmap='gray')
        axs[row, 2].set_title('Prediction')
        axs[row, 2].axis('off')

    plt.tight_layout()
    plt.show()

show_random_predictions(net, test_dataset, num_samples=6, threshold=THRESHOLD)

In [ ]:
# 9) Optional: run inference on one custom image (no ground-truth required)
# Set CUSTOM_IMAGE_PATH to an image file, then run this cell.
CUSTOM_IMAGE_PATH = None

@torch.no_grad()
def predict_single_image(model, image_path, image_size=512, threshold=0.5):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f'Image not found: {image_path}')

    img = Image.open(image_path).convert('RGB').resize((image_size, image_size), Image.BILINEAR)
    img_np = np.asarray(img, dtype=np.float32) / 255.0

    img_t = torch.from_numpy(img_np).permute(2, 0, 1).contiguous()
    norm_img_t = (img_t - MEAN) / STD

    logits = model(norm_img_t.unsqueeze(0).to(device))
    prob = torch.sigmoid(logits).squeeze().cpu().numpy()
    pred = (prob > threshold).astype(np.float32)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(img_np)
    plt.title('Input')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(prob, cmap='viridis')
    plt.title('Predicted Probability')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(pred, cmap='gray')
    plt.title('Predicted Binary Mask')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

if CUSTOM_IMAGE_PATH is None:
    print('Set CUSTOM_IMAGE_PATH to run single-image inference.')
else:
    predict_single_image(net, CUSTOM_IMAGE_PATH, image_size=IMAGE_SIZE, threshold=THRESHOLD)